# Inspeção Visual de Peças de Fundição Metálica

Mini-projeto de visão computacional + CNN para classificação binária de peças fundidas
(**OK** vs **Defeituosa**), usando o dataset *Casting Product Image Data for Quality Inspection*.

O fluxo segue seis sprints industriais: ingestão → OpenCV clássico → morfologia →
pipeline Keras → CNN → auditoria de performance.

## Sprint 1 — Configuração e ingestão dos dados

Objetivo: ambiente reprodutível e dataset acessível localmente em `casting_data/`
(`def_front/` e `ok_front/`). Sem seed fixa, qualquer comparação entre execuções
(treino, split, augmentation) fica inválida em auditoria.

In [ ]:
from pathlib import Path
import random
import zipfile

import numpy as np
import tensorflow as tf

# Mesma seed em NumPy, Python e TF: o split 80/20 e o shuffle do dataset
# precisam coincidir entre notebooks do time e entre reexecuções na fábrica.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Determinismo "melhor esforço" no TF (não elimina 100% da variação em GPU,
# mas reduz ruído na comparação de hiperparâmetros).
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"TensorFlow {tf.__version__} | seed={SEED}")

In [ ]:
# Caminhos do projeto
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "casting_data"
DRIVE_FILE_ID = "1NZOjCHDRrpn7PmbFKVqegUP5arfdXHKK"
ZIP_PATH = ROOT / "casting_dataset.zip"


def contar_imagens(pasta: Path) -> int:
    return sum(1 for p in pasta.rglob("*") if p.suffix.lower() in {".jpeg", ".jpg", ".png"})


def dataset_pronto(data_dir: Path) -> bool:
    """Exige as duas classes com pelo menos uma imagem cada (ignora .gitkeep vazio)."""
    d_def, d_ok = data_dir / "def_front", data_dir / "ok_front"
    return (
        d_def.is_dir()
        and d_ok.is_dir()
        and contar_imagens(d_def) > 0
        and contar_imagens(d_ok) > 0
    )


def baixar_dataset_drive(file_id: str, destino_zip: Path) -> None:
    # gdown trata a confirmação de download grande do Drive; requests puro falha com HTML de aviso.
    import gdown

    url = f"https://drive.google.com/uc?id={file_id}"
    print("Baixando dataset do Google Drive...")
    gdown.download(url, str(destino_zip), quiet=False)


def extrair_e_normalizar(zip_path: Path, data_dir: Path) -> None:
    """Extrai o zip e acomoda pastas caso o arquivo venha com um nível extra de diretório."""
    data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)

    # Alguns dumps trazem casting_data/casting_data/... — sobe um nível se necessário.
    nested = data_dir / "casting_data"
    if nested.is_dir() and not dataset_pronto(data_dir):
        for item in nested.iterdir():
            alvo = data_dir / item.name
            if not alvo.exists():
                item.rename(alvo)

    # Caso as classes estejam na raiz do projeto (legado), move para casting_data/
    for classe in ("def_front", "ok_front"):
        origem = ROOT / classe
        destino = data_dir / classe
        if origem.is_dir() and not destino.exists():
            origem.rename(destino)


if dataset_pronto(DATA_DIR):
    print(f"Dataset local encontrado em: {DATA_DIR}")
else:
    if not ZIP_PATH.exists():
        baixar_dataset_drive(DRIVE_FILE_ID, ZIP_PATH)
    extrair_e_normalizar(ZIP_PATH, DATA_DIR)
    if not dataset_pronto(DATA_DIR):
        raise FileNotFoundError(
            "Após o download, não achei def_front/ e ok_front/ em casting_data/. "
            "Confira a estrutura do arquivo do Drive."
        )
    print(f"Dataset preparado em: {DATA_DIR}")

n_def = contar_imagens(DATA_DIR / "def_front")
n_ok = contar_imagens(DATA_DIR / "ok_front")
print(f"Classes | def_front={n_def} | ok_front={n_ok} | total={n_def + n_ok}")
print(
    "Desbalanceamento leve é esperado neste dataset; a métrica accuracy "
    "sozinha pode mascarar falhas na classe minoritária — por isso olhamos as curvas na Sprint 6."
)